In [ ]:
import json
import numpy as np
import os

class SyntheticDatasetGenerator:
    def __init__(self, config_path):
        self.config = self.load_config(config_path)
        self.save_path = os.path.join("data", self.config["save_name"])
        if not os.path.exists(self.save_path):
            os.makedirs(self.save_path)

    def load_config(self, config_path):
        with open(config_path, 'r') as f:
            return json.load(f)

    def generate_feature(self, wave_type, f_support, f_base):
        x_feature = np.sin(np.linspace(0, 2 * np.pi * f_base, self.config["n_points"])).reshape(-1, 1)
        x_feature *= 0.5
        start_idx = np.random.randint(0, self.config["n_points"] - self.config["n_support"])
        
        if wave_type == "sine":
            x_tmp = np.sin(np.linspace(0, 2 * np.pi * f_support, self.config["n_points"])).reshape(-1, 1)
            x_feature[start_idx:start_idx + self.config["n_support"], 0] += x_tmp[start_idx:start_idx + self.config["n_support"], 0]
        elif wave_type == "square":
            x_tmp = np.sign(np.sin(np.linspace(0, 2 * np.pi * f_support, self.config["n_points"]))).reshape(-1, 1)
            x_feature[start_idx:start_idx + self.config["n_support"], 0] += x_tmp[start_idx:start_idx + self.config["n_support"], 0]
        elif wave_type == "line":
            x_feature[start_idx:start_idx + self.config["n_support"], 0] += np.zeros(self.config["n_support"])
        else:
            raise ValueError("wave must be one of sine, square, line")
        
        return x_feature, start_idx

    def generate_sample(self):
        dict_all = {}
        dict_all["signal"] = np.zeros((self.config["n_points"], self.config["n_feature"]))
        idx_features = np.random.permutation(np.arange(self.config["n_feature"]))
        f_sine_sum = 0
        
        for enum, idx_feature in enumerate(idx_features):
            f_base = np.random.randint(self.config["f_base"]["min"], self.config["f_base"]["max"] + 1)
            f_support = np.random.randint(self.config["f_sin"]["min"], self.config["f_sin"]["max"] + 1)
            
            if enum < 2:
                f_sine_sum += f_support
                x_tmp, _ = self.generate_feature("sine", f_support, f_base)
            else:
                wave_type = np.random.choice(["line", "square"])
                x_tmp, _ = self.generate_feature(wave_type, f_support, f_base)
            
            dict_all["signal"][:, idx_feature] = x_tmp.squeeze()
        
        # Class definition based on f_sine_sum
        dict_all["target"] = np.where(f_sine_sum <= np.quantile([np.random.randint(self.config["f_sin"]["min"], self.config["f_sin"]["max"] + 1) + np.random.randint(self.config["f_sin"]["min"], self.config["f_sin"]["max"] + 1) for _ in range(10000)], self.config["quantile_class"][0]), 0, 1)
        
        return dict_all

    def generate_dataset(self):
        for i in range(self.config["nb_simulation"]):
            sample = self.generate_sample()
            np.save(os.path.join(self.save_path, f"sample_{i}.npy"), sample)




In [7]:
generator = SyntheticDatasetGenerator("config_synthetic_small.json")
generator.generate_dataset()

In [8]:
import json
import numpy as np
import os

class SyntheticDatasetGenerator:
    def __init__(self, config_path):
        self.config = self.load_config(config_path)
        self.save_path = os.path.join("data_small", self.config["save_name"])
        if not os.path.exists(self.save_path):
            os.makedirs(self.save_path)

    def load_config(self, config_path):
        with open(config_path, 'r') as f:
            return json.load(f)

    def generate_feature(self, wave_type, f_support, f_base):
        x_feature = np.sin(np.linspace(0, 2 * np.pi * f_base, self.config["n_points"])).reshape(-1, 1)
        x_feature *= 0.5
        start_idx = np.random.randint(0, self.config["n_points"] - self.config["n_support"])
        
        if wave_type == "sine":
            x_tmp = np.sin(np.linspace(0, 2 * np.pi * f_support, self.config["n_points"])).reshape(-1, 1)
            x_feature[start_idx:start_idx + self.config["n_support"], 0] += x_tmp[start_idx:start_idx + self.config["n_support"], 0]
        elif wave_type == "square":
            x_tmp = np.sign(np.sin(np.linspace(0, 2 * np.pi * f_support, self.config["n_points"]))).reshape(-1, 1)
            x_feature[start_idx:start_idx + self.config["n_support"], 0] += x_tmp[start_idx:start_idx + self.config["n_support"], 0]
        elif wave_type == "line":
            x_feature[start_idx:start_idx + self.config["n_support"], 0] += np.zeros(self.config["n_support"])
        else:
            raise ValueError("wave must be one of sine, square, line")
        
        return x_feature, start_idx

    def generate_sample(self):
        dict_all = {}
        dict_all["signal"] = np.zeros((self.config["n_points"], self.config["n_feature"]))
        idx_features = np.random.permutation(np.arange(self.config["n_feature"]))
        f_sine_sum = 0
        
        # Store metadata
        dict_all["metadata"] = {
            "sine_feature_indices": [],
            "start_indices": [],
            "frequencies": [],
            "threshold": None
        }
        
        for enum, idx_feature in enumerate(idx_features):
            f_base = np.random.randint(self.config["f_base"]["min"], self.config["f_base"]["max"] + 1)
            f_support = np.random.randint(self.config["f_sin"]["min"], self.config["f_sin"]["max"] + 1)
            
            if enum < 2:
                f_sine_sum += f_support
                x_tmp, start_idx = self.generate_feature("sine", f_support, f_base)
                dict_all["signal"][:, idx_feature] = x_tmp.squeeze()
                
                # Store metadata for sine features
                dict_all["metadata"]["sine_feature_indices"].append(idx_feature)
                dict_all["metadata"]["start_indices"].append(start_idx)
                dict_all["metadata"]["frequencies"].append(f_support)
            else:
                wave_type = np.random.choice(["line", "square"])
                x_tmp, start_idx = self.generate_feature(wave_type, f_support, f_base)
                dict_all["signal"][:, idx_feature] = x_tmp.squeeze()
        
        # Class definition based on f_sine_sum
        threshold = np.quantile([np.random.randint(self.config["f_sin"]["min"], self.config["f_sin"]["max"] + 1) + np.random.randint(self.config["f_sin"]["min"], self.config["f_sin"]["max"] + 1) for _ in range(10000)], self.config["quantile_class"][0])
        dict_all["target"] = np.where(f_sine_sum <= threshold, 0, 1)
        dict_all["metadata"]["threshold"] = threshold
        
        return dict_all

    def generate_dataset(self):
        for i in range(self.config["nb_simulation"]):
            sample = self.generate_sample()
            np.save(os.path.join(self.save_path, f"sample_{i}.npy"), sample["signal"])
            np.save(os.path.join(self.save_path, f"target_{i}.npy"), sample["target"])
            with open(os.path.join(self.save_path, f"metadata_{i}.json"), 'w') as f:
                json.dump({k: [int(x) for x in v] if isinstance(v, list) else v for k, v in sample["metadata"].items()}, f)


In [9]:
generator = SyntheticDatasetGenerator("config_synthetic.json")
generator.generate_dataset()